# BuzzSpot - YOLO26s 24ep + hoverfly crop augmentation

## 1. Install


In [ ]:
!pip install -q ultralytics pycocotools tqdm

## 2. Mount Drive and locate dataset files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
from pathlib import Path

zip_hits = glob.glob("/content/drive/MyDrive/**/BuzzSet_challenge.zip", recursive=True)
assert zip_hits, "BuzzSet_challenge.zip not found under MyDrive"
ZIP_PATH = zip_hits[0]

tar_hits = glob.glob("/content/drive/MyDrive/**/19557529600-BuzzSet_challenge_testphase.tar", recursive=True)
if not tar_hits:
    tar_hits = glob.glob("/content/drive/MyDrive/**/*BuzzSet_challenge_testphase*.tar*", recursive=True)
assert tar_hits, "test-phase tar file not found under MyDrive"
TAR_PATH = tar_hits[0]

print("old train/valid zip:", ZIP_PATH)
print("new test-phase tar:", TAR_PATH)


Mounted at /content/drive
old train/valid zip: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/BuzzSet_challenge.zip
new test-phase tar: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/19557529600-BuzzSet_challenge_testphase.tar


## 3. Config


In [ ]:
from pathlib import Path
import shutil

# Main switch:
# - True: train again, then save best/last/results to Drive
# - False: skip training and restore saved best.pt from Drive
ENABLE_TRAINING = True

MODEL_NAME = "yolo26s.pt"
MODEL_TAG = "yolo26s_24ep_rare_os_hover1x_hfcrops2"
RUN_NAME = "buzz_yolo26s_24ep_rare_os_hover1x_hfcrops2"

EPOCHS = 24
CLOSE_MOSAIC = 3
IMGSZ = 1536
BATCH = 4

CONF = 0.01
IOU = 0.60
MAX_DET = 100

LOCAL_DATA_DIR = Path("/content/buzz")
LOCAL_WEIGHTS = Path("/content/best.pt")

PROJECT_DIR = Path("/content/drive/MyDrive/BuzzSpot")
WEIGHTS_DIR = PROJECT_DIR / "weights"
DRIVE_RUNS_DIR = PROJECT_DIR / "runs_oversampling"
SUBMISSIONS_DIR = PROJECT_DIR / "submissions"

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_BEST_WEIGHTS = WEIGHTS_DIR / f"{MODEL_TAG}_best.pt"
DRIVE_LAST_WEIGHTS = WEIGHTS_DIR / f"{MODEL_TAG}_last.pt"
DRIVE_RESULTS_CSV = WEIGHTS_DIR / f"{MODEL_TAG}_results.csv"

CONF_TAG = f"conf{int(CONF * 1000):03d}"
LOCAL_PRED_PATH = Path("/content/predictions.json")
LOCAL_ZIP_OUT = Path(f"/content/submission_{MODEL_TAG}_{CONF_TAG}.zip")
DRIVE_PRED_PATH = SUBMISSIONS_DIR / f"predictions_{MODEL_TAG}_{CONF_TAG}.json"
DRIVE_ZIP_OUT = SUBMISSIONS_DIR / f"submission_{MODEL_TAG}_{CONF_TAG}.zip"

CLASS_NAMES = ["bee", "bumblebee", "hoverfly", "moth"]
CATEGORY_NAME_TO_ID = {name: i + 1 for i, name in enumerate(CLASS_NAMES)}

# YOLO class ids: 0 bee, 1 bumblebee, 2 hoverfly, 3 moth
# Full-frame image-level oversampling. Hoverfly is intentionally 1x because
# prior run had poor hoverfly precision; we add separate hoverfly-focused crops below.
CLASS_MULTIPLIER = {
    0: 1,  # bee
    1: 4,  # bumblebee
    2: 1,  # hoverfly
    3: 5,  # moth
}

# Extra hoverfly-focused crop augmentation.
# For every train hoverfly annotation, create 2 crop images around it.
ENABLE_HOVERFLY_CROPS = True
HOVERFLY_CROPS_PER_ANN = 2
HOVERFLY_CROP_SIZES = [640, 768, 896]
HOVERFLY_CROP_SEED = 0
HOVERFLY_CROP_MIN_KEEP_FRAC = 0.25  # keep other boxes if at least 25% remains in crop
HOVERFLY_YOLO_CLASS_ID = 2          # category_id 3 -> YOLO cls 2


def restore_best_weights_to_local():
    """
    Restores the saved Drive checkpoint into /content/best.pt.
    This is what lets inference/validation work after runtime deletion.
    """
    candidates = [
        DRIVE_BEST_WEIGHTS,
        DRIVE_RUNS_DIR / RUN_NAME / "weights" / "best.pt",
    ]

    for p in candidates:
        if p.exists():
            shutil.copy(p, LOCAL_WEIGHTS)
            print("Restored best.pt from Drive:", p)
            print("Local weights:", LOCAL_WEIGHTS)
            return LOCAL_WEIGHTS

    if LOCAL_WEIGHTS.exists():
        print("WARNING: Drive backup not found, but local best.pt exists:", LOCAL_WEIGHTS)
        return LOCAL_WEIGHTS

    raise FileNotFoundError(
        "No saved best.pt found. Set ENABLE_TRAINING = True and run all cells once."
    )


def backup_training_outputs_to_drive():
    """
    Copies best.pt, last.pt, and results.csv from the Drive run folder
    into stable Drive backup paths.
    """
    run_dir = DRIVE_RUNS_DIR / RUN_NAME
    best_path = run_dir / "weights" / "best.pt"
    last_path = run_dir / "weights" / "last.pt"
    results_csv = run_dir / "results.csv"

    assert best_path.exists(), f"best.pt not found at {best_path}"

    shutil.copy(best_path, LOCAL_WEIGHTS)
    shutil.copy(best_path, DRIVE_BEST_WEIGHTS)
    print("Copied best weights to local:", LOCAL_WEIGHTS)
    print("Backed up best weights to Drive:", DRIVE_BEST_WEIGHTS)

    if last_path.exists():
        shutil.copy(last_path, DRIVE_LAST_WEIGHTS)
        print("Backed up last weights to Drive:", DRIVE_LAST_WEIGHTS)

    if results_csv.exists():
        shutil.copy(results_csv, DRIVE_RESULTS_CSV)
        print("Backed up results.csv to Drive:", DRIVE_RESULTS_CSV)

print("ENABLE_TRAINING:", ENABLE_TRAINING)
print("model:", MODEL_NAME)
print("epochs:", EPOCHS)
print("close_mosaic:", CLOSE_MOSAIC)
print("imgsz:", IMGSZ)
print("batch:", BATCH)
print("run:", RUN_NAME)
print("Drive best weights:", DRIVE_BEST_WEIGHTS)
print("Drive zip output:", DRIVE_ZIP_OUT)
print("oversampling:", {CLASS_NAMES[k]: v for k, v in CLASS_MULTIPLIER.items()})
print("hoverfly crops enabled:", ENABLE_HOVERFLY_CROPS)
print("hoverfly crops per annotation:", HOVERFLY_CROPS_PER_ANN)
print("hoverfly crop sizes:", HOVERFLY_CROP_SIZES)


ENABLE_TRAINING: True
model: yolo26s.pt
epochs: 24
close_mosaic: 3
imgsz: 1536
batch: 4
run: buzz_yolo26s_24ep_rare_os_hover1x_hfcrops2
Drive best weights: /content/drive/MyDrive/BuzzSpot/weights/yolo26s_24ep_rare_os_hover1x_hfcrops2_best.pt
Drive zip output: /content/drive/MyDrive/BuzzSpot/submissions/submission_yolo26s_24ep_rare_os_hover1x_hfcrops2_conf010.zip
oversampling: {'bee': 1, 'bumblebee': 4, 'hoverfly': 1, 'moth': 5}
hoverfly crops enabled: True
hoverfly crops per annotation: 2
hoverfly crop sizes: [640, 768, 896]


## 4. Build YOLO dataset + hoverfly crop augmentation + oversampled train list

In [ ]:
import json, zipfile, tarfile, shutil, random
from pathlib import Path
from collections import defaultdict, Counter
from PIL import Image

# clean old local dataset
if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)

for d in [
    "images/train", "images/val", "images/test_testphase", "images/train_hoverfly_crops",
    "labels/train", "labels/val", "labels/train_hoverfly_crops", "annotations"
]:
    (LOCAL_DATA_DIR / d).mkdir(parents=True, exist_ok=True)

zf = zipfile.ZipFile(ZIP_PATH)
tf = tarfile.open(TAR_PATH, "r:*")

zip_names = set(zf.namelist())
tar_members = {m.name: m for m in tf.getmembers() if m.isfile()}
tar_names = set(tar_members.keys())


def find_zip(rel):
    rel = rel.lstrip("/")
    for cand in (rel, f"BuzzSet_challenge/{rel}"):
        if cand in zip_names:
            return cand
    suffix = "/" + rel
    hits = [n for n in zip_names if n.endswith(suffix)]
    return hits[0] if hits else None


def find_tar(rel):
    rel = rel.lstrip("/")
    for cand in (rel, f"BuzzSet_challenge/{rel}", f"BuzzSet_challenge_testphase/{rel}"):
        if cand in tar_names:
            return cand
    suffix = "/" + rel
    hits = [n for n in tar_names if n.endswith(suffix)]
    return hits[0] if hits else None


def load_tar_json(fname):
    p = find_tar(f"annotations/{fname}") or find_tar(fname)
    assert p is not None, f"{fname} not found in tar"
    with tf.extractfile(tar_members[p]) as f:
        obj = json.load(f)
    out = LOCAL_DATA_DIR / "annotations" / fname
    out.write_text(json.dumps(obj))
    print("loaded current annotation:", fname, "from", p)
    return obj


def flat_name(file_name):
    return file_name.replace("/", "__")


def write_yolo_label(label_path, anns, W, H):
    lines = []
    for ann in anns:
        x, y, w, h = ann["bbox"]
        cls = int(ann["category_id"]) - 1

        # clip just in case
        x = max(0.0, min(float(x), W - 1))
        y = max(0.0, min(float(y), H - 1))
        w = max(0.0, min(float(w), W - x))
        h = max(0.0, min(float(h), H - y))

        if w < 1 or h < 1:
            continue

        xc = (x + w / 2) / W
        yc = (y + h / 2) / H
        ww = w / W
        hh = h / H

        lines.append(f"{cls} {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}")

    label_path.write_text("\n".join(lines))


def extract_annotated_split(coco, zip_img_dir, out_split):
    by_img = defaultdict(list)
    for ann in coco.get("annotations", []):
        by_img[int(ann["image_id"])].append(ann)

    images_by_id = {int(im["id"]): im for im in coco["images"]}
    image_ids = sorted(by_img.keys())

    written_images = []
    count_images = 0
    count_boxes = 0
    class_counts = Counter()

    for iid in image_ids:
        im = images_by_id[iid]
        src = find_zip(f"{zip_img_dir}/{im['file_name']}") or find_zip(im["file_name"])
        assert src is not None, f"missing image in old zip: {zip_img_dir}/{im['file_name']}"

        out_name = flat_name(im["file_name"])
        img_dst = LOCAL_DATA_DIR / "images" / out_split / out_name
        lbl_dst = LOCAL_DATA_DIR / "labels" / out_split / (Path(out_name).stem + ".txt")

        with zf.open(src) as s, open(img_dst, "wb") as d:
            shutil.copyfileobj(s, d)

        anns = by_img[iid]
        write_yolo_label(lbl_dst, anns, int(im["width"]), int(im["height"]))

        written_images.append(img_dst)
        count_images += 1
        count_boxes += len(anns)
        for ann in anns:
            class_counts[int(ann["category_id"])] += 1

    print(out_split, "images:", count_images, "boxes:", count_boxes, "class counts:", dict(class_counts))
    print(out_split, "named counts:", {CLASS_NAMES[k-1]: class_counts[k] for k in range(1, 5)})
    return written_images


def extract_test_keyframes(test_coco):
    keyframe_images = [
        im for im in test_coco["images"]
        if im.get("is_keyframe") in [True, 1, "true", "True"]
    ]

    flat_to_id = {}

    for im in keyframe_images:
        src = find_tar(f"test_testphase/{im['file_name']}") or find_tar(im["file_name"])
        assert src is not None, f"missing test image in tar: test_testphase/{im['file_name']}"

        out_name = flat_name(im["file_name"])
        dst = LOCAL_DATA_DIR / "images" / "test_testphase" / out_name

        with tf.extractfile(tar_members[src]) as s, open(dst, "wb") as d:
            shutil.copyfileobj(s, d)

        flat_to_id[out_name] = int(im["id"])

    keyframe_paths = [
        str(LOCAL_DATA_DIR / "images" / "test_testphase" / flat_name(im["file_name"]))
        for im in keyframe_images
    ]

    print("test_testphase keyframes:", len(keyframe_paths))
    return keyframe_images, flat_to_id, keyframe_paths


def yolo_label_path_from_image_path(img_path: Path) -> Path:
    s = str(img_path)
    assert "/images/" in s, f"Cannot infer label path from image path: {img_path}"
    return Path(s.replace("/images/", "/labels/", 1)).with_suffix(".txt")


def classes_in_label(label_path: Path):
    classes = set()
    if not label_path.exists():
        return classes
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                classes.add(int(float(parts[0])))
    return classes


def count_instances(label_path: Path):
    counts = Counter()
    if not label_path.exists():
        return counts
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                counts[int(float(parts[0]))] += 1
    return counts


def create_hoverfly_crop_labels(anns, crop_left, crop_top, crop_w, crop_h):
    crop_right = crop_left + crop_w
    crop_bottom = crop_top + crop_h
    records = []

    for ann in anns:
        x, y, w, h = map(float, ann["bbox"])
        x1, y1, x2, y2 = x, y, x + w, y + h

        ix1 = max(x1, crop_left)
        iy1 = max(y1, crop_top)
        ix2 = min(x2, crop_right)
        iy2 = min(y2, crop_bottom)

        iw = ix2 - ix1
        ih = iy2 - iy1
        if iw < 1 or ih < 1:
            continue

        keep_frac = (iw * ih) / max(1.0, w * h)
        if keep_frac < HOVERFLY_CROP_MIN_KEEP_FRAC:
            continue

        cls = int(ann["category_id"]) - 1
        xc = (ix1 - crop_left + iw / 2) / crop_w
        yc = (iy1 - crop_top + ih / 2) / crop_h
        ww = iw / crop_w
        hh = ih / crop_h
        records.append((cls, f"{cls} {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}"))

    return records


def make_hoverfly_crops(train_coco):
    if not ENABLE_HOVERFLY_CROPS:
        print("hoverfly crop augmentation disabled")
        return []

    rng = random.Random(HOVERFLY_CROP_SEED)
    crop_img_dir = LOCAL_DATA_DIR / "images" / "train_hoverfly_crops"
    crop_lbl_dir = LOCAL_DATA_DIR / "labels" / "train_hoverfly_crops"
    crop_img_dir.mkdir(parents=True, exist_ok=True)
    crop_lbl_dir.mkdir(parents=True, exist_ok=True)

    by_img = defaultdict(list)
    for ann in train_coco.get("annotations", []):
        by_img[int(ann["image_id"])].append(ann)

    images_by_id = {int(im["id"]): im for im in train_coco["images"]}

    crop_paths = []
    crop_instance_counts = Counter()
    hoverfly_ann_count = 0
    skipped = 0

    for iid in sorted(by_img.keys()):
        anns = by_img[iid]
        hoverfly_anns = [a for a in anns if int(a["category_id"]) == 3]
        if not hoverfly_anns:
            continue

        im = images_by_id[iid]
        W, H = int(im["width"]), int(im["height"])
        src_img = LOCAL_DATA_DIR / "images" / "train" / flat_name(im["file_name"])
        assert src_img.exists(), f"missing extracted train image for crops: {src_img}"

        with Image.open(src_img) as img:
            img = img.convert("RGB")

            for target_ann in hoverfly_anns:
                hoverfly_ann_count += 1
                x, y, w, h = map(float, target_ann["bbox"])
                cx = x + w / 2
                cy = y + h / 2

                for j in range(HOVERFLY_CROPS_PER_ANN):
                    size = int(rng.choice(HOVERFLY_CROP_SIZES))
                    crop_w = min(size, W)
                    crop_h = min(size, H)

                    # Jitter the crop center while keeping the target hoverfly inside the crop.
                    max_jx = max(0.0, crop_w / 2 - w / 2 - 2)
                    max_jy = max(0.0, crop_h / 2 - h / 2 - 2)
                    jx = rng.uniform(-0.25 * crop_w, 0.25 * crop_w)
                    jy = rng.uniform(-0.25 * crop_h, 0.25 * crop_h)
                    jx = max(-max_jx, min(max_jx, jx))
                    jy = max(-max_jy, min(max_jy, jy))

                    left = int(round(cx + jx - crop_w / 2))
                    top = int(round(cy + jy - crop_h / 2))
                    left = max(0, min(left, W - crop_w))
                    top = max(0, min(top, H - crop_h))

                    records = create_hoverfly_crop_labels(anns, left, top, crop_w, crop_h)
                    if not any(cls == HOVERFLY_YOLO_CLASS_ID for cls, _ in records):
                        skipped += 1
                        continue

                    stem = Path(flat_name(im["file_name"])).stem
                    crop_name = f"{stem}__hfann{int(target_ann.get('id', hoverfly_ann_count))}__crop{j}__{crop_w}x{crop_h}.jpg"
                    crop_img_path = crop_img_dir / crop_name
                    crop_lbl_path = crop_lbl_dir / (Path(crop_name).stem + ".txt")

                    img.crop((left, top, left + crop_w, top + crop_h)).save(crop_img_path, quality=95)
                    crop_lbl_path.write_text("\n".join(line for _, line in records))
                    crop_paths.append(crop_img_path)
                    crop_instance_counts.update(cls for cls, _ in records)

    print("\nhoverfly crop augmentation summary")
    print("hoverfly train annotations:", hoverfly_ann_count)
    print("requested crops:", hoverfly_ann_count * HOVERFLY_CROPS_PER_ANN)
    print("created crop images:", len(crop_paths))
    print("skipped crops:", skipped)
    print("crop instance counts:", {CLASS_NAMES[k]: crop_instance_counts[k] for k in range(4)})
    print("crop image dir:", crop_img_dir)
    print("crop label dir:", crop_lbl_dir)
    return crop_paths


def build_oversampled_train_txt(train_image_paths, hoverfly_crop_paths=None):
    hoverfly_crop_paths = hoverfly_crop_paths or []
    oversampled_lines = []
    image_multiplier_counts = Counter()
    class_image_counts = Counter()
    original_instance_counts = Counter()
    effective_instance_counts = Counter()
    crop_instance_counts = Counter()

    # Full-frame images get image-level oversampling.
    for img_path in train_image_paths:
        lp = yolo_label_path_from_image_path(img_path)
        cls_set = classes_in_label(lp)
        inst_counts = count_instances(lp)

        for c in cls_set:
            class_image_counts[c] += 1
        original_instance_counts.update(inst_counts)

        mult = max([CLASS_MULTIPLIER.get(c, 1) for c in cls_set] or [1])
        image_multiplier_counts[mult] += 1

        for _ in range(mult):
            oversampled_lines.append(str(img_path.resolve()))

        for c, n in inst_counts.items():
            effective_instance_counts[c] += n * mult

    # Hoverfly-focused crops are added once each. They are already targeted augmentation.
    for img_path in hoverfly_crop_paths:
        lp = yolo_label_path_from_image_path(img_path)
        inst_counts = count_instances(lp)
        crop_instance_counts.update(inst_counts)
        effective_instance_counts.update(inst_counts)
        oversampled_lines.append(str(img_path.resolve()))

    train_txt = LOCAL_DATA_DIR / "train_oversampled.txt"
    train_txt.write_text("\n".join(oversampled_lines) + "\n")

    print("\noversampling + crop summary")
    print("original full-frame train images:", len(train_image_paths))
    print("hoverfly crop images added:", len(hoverfly_crop_paths))
    print("total train entries:", len(oversampled_lines))
    print("full-frame slowdown factor only:", round((len(oversampled_lines) - len(hoverfly_crop_paths)) / len(train_image_paths), 3))
    print("total factor including crops:", round(len(oversampled_lines) / len(train_image_paths), 3))
    print("image multiplier counts:", dict(sorted(image_multiplier_counts.items())))

    print("\nclass image counts before oversampling/full-frame only:")
    for k, name in enumerate(CLASS_NAMES):
        print(f"  {name:10s}: {class_image_counts[k]}")

    print("\ninstance counts before oversampling/full-frame only:")
    for k, name in enumerate(CLASS_NAMES):
        print(f"  {name:10s}: {original_instance_counts[k]}")

    print("\ninstances added inside hoverfly crops:")
    for k, name in enumerate(CLASS_NAMES):
        print(f"  {name:10s}: {crop_instance_counts[k]}")

    print("\neffective instance counts after oversampling + crops:")
    for k, name in enumerate(CLASS_NAMES):
        print(f"  {name:10s}: {effective_instance_counts[k]}")

    print("\nwrote:", train_txt)
    return train_txt

# Current annotations come from the new tar.
# Train/valid images come from the old zip.
# Test images/json come from the new tar.
train = load_tar_json("train.json")
valid = load_tar_json("valid.json")
test = load_tar_json("test_testphase.json")

train_image_paths = extract_annotated_split(train, "train", "train")
valid_image_paths = extract_annotated_split(valid, "valid", "val")
keyframe_images, flat_to_id, keyframe_paths = extract_test_keyframes(test)

hoverfly_crop_paths = make_hoverfly_crops(train)
OVERSAMPLED_TRAIN_TXT = build_oversampled_train_txt(train_image_paths, hoverfly_crop_paths)

zf.close()
tf.close()

RAW_DATA_YAML = LOCAL_DATA_DIR / "data_raw.yaml"
RAW_DATA_YAML.write_text(
    f"path: {LOCAL_DATA_DIR}\n"
    "train: images/train\n"
    "val: images/val\n"
    "nc: 4\n"
    "names: ['bee', 'bumblebee', 'hoverfly', 'moth']\n"
)

DATA_YAML = LOCAL_DATA_DIR / "data_oversampled_hfcrops.yaml"
DATA_YAML.write_text(
    f"path: {LOCAL_DATA_DIR}\n"
    "train: train_oversampled.txt\n"
    "val: images/val\n"
    "nc: 4\n"
    "names: ['bee', 'bumblebee', 'hoverfly', 'moth']\n"
)

print("\nraw data yaml:")
print(RAW_DATA_YAML.read_text())

print("oversampled + hoverfly crops data yaml used for training:")
print(DATA_YAML.read_text())


loaded current annotation: train.json from BuzzSpot_testphase/annotations/train.json
loaded current annotation: valid.json from BuzzSpot_testphase/annotations/valid.json
loaded current annotation: test_testphase.json from BuzzSpot_testphase/annotations/test_testphase.json
train images: 5275 boxes: 10884 class counts: {1: 8677, 3: 1753, 2: 237, 4: 217}
train named counts: {'bee': 8677, 'bumblebee': 237, 'hoverfly': 1753, 'moth': 217}
val images: 932 boxes: 1116 class counts: {1: 934, 2: 30, 3: 95, 4: 57}
val named counts: {'bee': 934, 'bumblebee': 30, 'hoverfly': 95, 'moth': 57}
test_testphase keyframes: 4763

hoverfly crop augmentation summary
hoverfly train annotations: 1753
requested crops: 3506
created crop images: 3506
skipped crops: 0
crop instance counts: {'bee': 1582, 'bumblebee': 25, 'hoverfly': 4184, 'moth': 0}
crop image dir: /content/buzz/images/train_hoverfly_crops
crop label dir: /content/buzz/labels/train_hoverfly_crops

oversampling + crop summary
original full-frame tra

## 5. Train YOLO26s with rare-class oversampling + hoverfly crops

In [ ]:
from ultralytics import YOLO
import torch, gc

if ENABLE_TRAINING:
    print("ENABLE_TRAINING=True -> training will run.")

    model = YOLO(MODEL_NAME)

    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=20,
        save_period=10,
        project=str(DRIVE_RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        seed=0,
        deterministic=True,
        cache="disk",
        workers=4,
        close_mosaic=CLOSE_MOSAIC,
    )

    backup_training_outputs_to_drive()

    gc.collect()
    torch.cuda.empty_cache()

else:
    print("ENABLE_TRAINING=False -> skipping training.")
    restore_best_weights_to_local()


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ENABLE_TRAINING=True -> training will run.
Ultralytics 8.4.81 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=3, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/buzz/data_oversampled_hfcrops.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=24, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torch

## 6. Validate best checkpoint


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

restore_best_weights_to_local()

model = YOLO(str(LOCAL_WEIGHTS))
print("classes:", model.names)

sweep_results = []

CONF_LIST = [0.001, 0.003, 0.005, 0.01]
IOU_LIST = [0.50, 0.60, 0.70]
MAX_DET_LIST = [100, 200]

for conf in CONF_LIST:
    for iou in IOU_LIST:
        for max_det in MAX_DET_LIST:
            print(f"\n=== conf={conf}, iou={iou}, max_det={max_det} ===")

            metrics = model.val(
                data=str(DATA_YAML),
                imgsz=IMGSZ,
                batch=1,
                split="val",
                conf=conf,
                iou=iou,          # this is NMS IoU threshold
                max_det=max_det,
                agnostic_nms=False,
                plots=False,
                verbose=False,
            )

            row = {
                "conf": conf,
                "iou": iou,
                "max_det": max_det,
                "map50_95": float(metrics.box.map),
                "map50": float(metrics.box.map50),
                "map75": float(metrics.box.map75),
            }

            # per-class AP50-95 if available
            for cls_idx, cls_name in model.names.items():
                row[f"ap_{cls_name}"] = float(metrics.box.maps[cls_idx])

            sweep_results.append(row)
            print(row)

df = pd.DataFrame(sweep_results).sort_values("map50_95", ascending=False)
display(df)

Restored best.pt from Drive: /content/drive/MyDrive/BuzzSpot/weights/yolo26s_24ep_rare_os_hover1x_hfcrops2_best.pt
Local weights: /content/best.pt
classes: {0: 'bee', 1: 'bumblebee', 2: 'hoverfly', 3: 'moth'}
Ultralytics 8.4.81 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO26s summary (fused): 122 layers, 9,466,728 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 172.4±15.4 MB/s, size: 628.9 KB)
val: Scanning /content/buzz/labels/val.cache... 932 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 932/932 325.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 932/932 46.5it/s 20.0s
                   all        932       1116      0.657      0.523       0.59      0.372
                   bee        841        934      0.849      0.864       0.91      0.507
             bumblebee         28         30      0.741      0.733      0.768      0.533
              

## 7. Full-frame inference on test_testphase keyframes

In [ ]:
import json, zipfile, gc, torch, shutil
from pathlib import Path
from ultralytics import YOLO

restore_best_weights_to_local()

model = YOLO(str(LOCAL_WEIGHTS))

try:
    del results
except:
    pass

gc.collect()
torch.cuda.empty_cache()

W, H = 1920, 1080
preds = []

for i, img_path in enumerate(keyframe_paths):
    with torch.inference_mode():
        r = model.predict(
            source=str(img_path),
            imgsz=IMGSZ,
            conf=CONF,
            iou=IOU,
            max_det=MAX_DET,
            batch=1,
            verbose=False
        )[0]

    fname = Path(r.path).name
    iid = flat_to_id.get(fname)

    if iid is not None:
        for b in r.boxes:
            x1, y1, x2, y2 = b.xyxy[0].tolist()

            x1 = max(0.0, min(x1, W - 1))
            y1 = max(0.0, min(y1, H - 1))
            x2 = max(0.0, min(x2, W))
            y2 = max(0.0, min(y2, H))

            w = x2 - x1
            h = y2 - y1

            if w >= 1 and h >= 1:
                preds.append({
                    "image_id": int(iid),
                    "category_id": int(b.cls[0]) + 1,
                    "bbox": [round(x1, 2), round(y1, 2), round(w, 2), round(h, 2)],
                    "score": round(float(b.conf[0]), 5),
                })

    del r

    if i % 100 == 0:
        print(i, "/", len(keyframe_paths), "preds:", len(preds))
        gc.collect()
        torch.cuda.empty_cache()

with open(LOCAL_PRED_PATH, "w") as f:
    json.dump(preds, f)

with zipfile.ZipFile(LOCAL_ZIP_OUT, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(LOCAL_PRED_PATH, arcname="predictions.json")

shutil.copy(LOCAL_PRED_PATH, DRIVE_PRED_PATH)
shutil.copy(LOCAL_ZIP_OUT, DRIVE_ZIP_OUT)

print("detections:", len(preds))
print("local predictions:", LOCAL_PRED_PATH)
print("local zip:", LOCAL_ZIP_OUT)
print("Drive predictions:", DRIVE_PRED_PATH)
print("Drive zip:", DRIVE_ZIP_OUT)


## 8. Validate submission zip before upload


In [ ]:
import json, zipfile
from pathlib import Path
from collections import Counter

p = json.load(open(LOCAL_PRED_PATH))
ids = {int(d["image_id"]) for d in p}
keyframe_ids = {int(im["id"]) for im in keyframe_images}

print("detections:", len(p))
print("distinct predicted images:", len(ids))
print("total keyframe images:", len(keyframe_ids))
print("predictions outside keyframes:", len(ids - keyframe_ids))
print("keyframes with no detections:", len(keyframe_ids - ids))
print("categories:", sorted({int(d["category_id"]) for d in p}))
print("class counts:", dict(Counter(int(d["category_id"]) for d in p)))

print("degenerate:", sum(1 for d in p if d["bbox"][2] < 1 or d["bbox"][3] < 1))
print("out of bounds:", sum(
    1 for d in p
    if d["bbox"][0] < -0.5
    or d["bbox"][1] < -0.5
    or d["bbox"][0] + d["bbox"][2] > 1920.5
    or d["bbox"][1] + d["bbox"][3] > 1080.5
))

with zipfile.ZipFile(LOCAL_ZIP_OUT) as z:
    contents = z.namelist()
    print("zip contents:", contents)

assert len(ids - keyframe_ids) == 0, "Submission has predictions outside keyframes"
assert sorted({int(d["category_id"]) for d in p}) and set(int(d["category_id"]) for d in p).issubset({1,2,3,4}), "Bad category ids"
assert all(d["bbox"][2] >= 1 and d["bbox"][3] >= 1 for d in p), "Degenerate boxes exist"
assert contents == ["predictions.json"], "Zip must contain exactly predictions.json"

print("\nUpload this zip:")
print(DRIVE_ZIP_OUT)


detections: 34819
distinct predicted images: 4649
total keyframe images: 4763
predictions outside keyframes: 0
keyframes with no detections: 114
categories: [1, 2, 3, 4]
class counts: {1: 25646, 3: 8587, 2: 405, 4: 181}
degenerate: 0
out of bounds: 0
zip contents: ['predictions.json']

Upload this zip:
/content/drive/MyDrive/BuzzSpot/submissions/submission_yolo26s_24ep_rare_os_hover1x_hfcrops2_conf010.zip
